# Loading Dataset

In [ ]:
import pandas as pd
df = pd.read_csv("Datasets/twcs/twcs.csv")
# print(df.tail())
print(df.columns)

# Filtering data for sprintcare

In [ ]:
selectedCompany = 'sprintcare'
data = df[
    (df['author_id'] == selectedCompany) |
    (df['text'].str.startswith('@' + selectedCompany))
]

print(data.columns)

# Datetime parser

In [ ]:
import datetime
def parse_date(date_str):
    return datetime.datetime.strptime(date_str, '%a %b %d %H:%M:%S %z %Y').strftime('%Y-%m-%d %H:%M:%S')

data['created_at'] = [parse_date(date_str) for date_str in data['created_at']]

# Formatting data for embeddings

In [ ]:
# sorting data for chronological order
data = data.sort_values(by='created_at')
# print(data.head())
authors = data.groupby(by='author_id')['author_id'].unique()

tagging = len('@' + selectedCompany)

conversations = []

for author in authors:
    conversation = data[
        (data['author_id'] == author[0]) |
        (data['text'].str.startswith('@' + author[0]))
    ]
    
    # print(conversation)
    conversation_string = ''
    for idx, row in conversation.iterrows():
        if row['inbound'] == True:
            conversation_string += "User: " + row['text'] + "\n"
        else:
            conversation_string += "Host: " + row['text'][len(author[0]) + 1:len(row['text'])] + "\n"
    
    conversations.append(conversation_string)
    
    
print(conversations[0])

In [ ]:
import json
filename = "Datasets/cleaned_conversations_" + selectedCompany + ".txt"

print("Saving conversations...")
with open(filename, 'w') as file:
    json.dump(conversations, file)

print("Conversations saved successfully !!")


# Reload from file
# with open(filename, 'r') as file:
#     loaded_list = json.load(file)

# print(loaded_list)

# Installation of sentence transformers for Embeddings 

In [ ]:
# install sentence transformer inside notebook
%pip install sentence-transformers

# Generating Embeddings for each conversation for faster retrieval during queries

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
def chunk_text(text, chunk_size=1000, overlap_pct=0.35):
    if not text: return []
    overlap_size = int(chunk_size * overlap_pct)
    step_size = chunk_size - overlap_size
    return [text[i:i + chunk_size] for i in range(0, len(text), step_size)]

rag_database = []

print("Embeddings Started...")
for original_index, row_text in enumerate(conversations):
    chunks = chunk_text(row_text, chunk_size=1000, overlap_pct=0.35)
    chunk_vectors = embedding_model.encode(chunks)
    
    # Store EACH chunk independently with its vector and metadata
    for chunk_text_str, vector in zip(chunks, chunk_vectors):
        rag_database.append({
            'original_index': original_index,
            'text': chunk_text_str,
            'embedding': vector
        })

# Save the structured database
np.save('rag_database.npy', rag_database)
print(f"Total searchable chunks generated: {len(rag_database)}")

# Installing dependencies for agent inside notebook

In [ ]:
%pip install langchain-ollama sentence-transformers numpy langchain

# Agent implementation begins

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

whole_embeddings = np.load("rag_database.npy", allow_pickle=True)
print(whole_embeddings.shape)
print(whole_embeddings[0].keys())
print(whole_embeddings[5]['text'])

doc_embeddings = [item['embedding'] for item in whole_embeddings]

(12975,)
dict_keys(['original_index', 'text', 'embedding'])
Host:  Hi, yes we do have a trade in deal. Just click the link https://t.co/0zzgy1hdHv for more information. - LS
User: @sprintcare My fakily and I have 2 iPhone 7s and 2 iPhone 7 pluses. Are those eligible for trade in for the 8 or X?
Host:  Yes, those devices are eligible for the trade in. - LS



# Tool for chunk retrieval 

In [4]:
from langchain_core.tools import tool

@tool
def search_existing_solution(query: str) -> str:
    """
    Searches the knowledge base for existing solutions. 
    Returns up to 2 chunks with >= 70% similarity to the query.
    """
    query_embedding = embedding_model.encode(query)
    
    # Calculate Cosine Similarity: dot(A, B) / (norm(A) * norm(B))
    norms = np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
    similarities = np.dot(doc_embeddings, query_embedding) / norms
    
    # Apply 70% (0.70) threshold
    valid_indices = np.where(similarities >= 0.70)[0]
    
    if len(valid_indices) == 0:
        return "NO_CHUNKS_FOUND"
    
    # Sort by descending similarity and grab top 2
    sorted_indices = valid_indices[np.argsort(similarities[valid_indices])[::-1]]
    top_2_indices = sorted_indices[:2]
    
    results = [f"- {whole_embeddings[i]['text']} (Score: {similarities[i]:.2f})" for i in top_2_indices]
    return "\n".join(results)

# Fetch test from tool

In [ ]:
print(search_existing_solution.invoke({"query" : "My family and I have 2 iPhone 7s and 2 iPhone 7 pluses. Are those eligible for trade in for the 8 or X"}))

# Intent Classifier

In [8]:
def classify_intent(user_message: str):
    intent_prompt = ChatPromptTemplate.from_messages([
            ("system", "Classify the customer message into one of these exact intents: [Network Issue, Billing, Device Upgrade, Cancellation, General Inquiry]. Return ONLY the intent name."),
            ("human", "{message}")
        ])
    return (intent_prompt | llm).invoke({"message": user_message}).content.strip()

# Intent classification from original messages for evaluation in further steps
 0. General Inquiry
 1. General Inquiry
 2. Network Issue
 3. Network Issue
 4. General Inquiry
 5. Device Upgrade
 6. Cancellation
 7. General Inquiry
 8. Network Issue
 9. Network Issue
 10. Network Issue

In [12]:
i = 0
for embedding in whole_embeddings:
    if i > 6:
        print(classify_intent(embedding['text']))
    i += 1
    if i > 10:
        break

General Inquiry
Network Issue
Network Issue
Network Issue


# Implementing agent

In [ ]:
llm = ChatOllama(model="llama3.1", temperature=0)

def run_agent(user_message: str):
    # Step A: Intent Classification
    intent_res = classify_intent(user_message=user_message)

    # Step B: Escalation Logic Policy
    escalate = False
    escalation_reason = None
    
    churn_keywords = ["cancel", "switch", "horrible", "worst", "lawyer", "attorney", "rip off"]
    if any(word in user_message.lower() for word in churn_keywords) or intent_res == "Cancellation":
        escalate = True
        escalation_reason = "High churn risk or explicit cancellation request detected."

    if escalate:
        return {
            "intent": intent_res,
            "action": "ESCALATE",
            "reason": escalation_reason,
            "reply": "We hate to see you go and want to make this right. I am escalating your case to a senior supervisor who will reach out shortly."
        }

    # Step C: RAG Grounded Reply Generation
    chunks = search_existing_solution.invoke({"query": user_message})
    if chunks == "NO_CHUNKS_FOUND":
        return {
            "intent": intent_res,
            "action": "ESCALATE",
            "reason": "No historical context or matching resolution found in database.",
            "reply": "I want to get this sorted out for you properly. Let me pass your details to our human support team."
        }

    reply_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful customer support agent for SprintCare. Draft a short, polite Twitter reply grounded strictly in the provided historical chunks."),
        ("human", "Historical Chunks:\n{chunks}\n\nCustomer Message: {message}")
    ])
    reply_res = (reply_prompt | llm).invoke({"chunks": chunks, "message": user_message}).content.strip()

    return {
        "intent": intent_res,
        "action": "AUTO_HANDLE",
        "reason": "Sufficient historical context available and low churn risk.",
        "reply": reply_res
    }

# Try Agent

In [14]:
print(run_agent("My family and I have 2 iPhone 7s and 2 iPhone 7 pluses. Are those eligible for trade in for the 8 or X")['reply'])

"Hi there! Yes, your iPhone 7s and 7 Pluses are eligible for trade in towards the iPhone 8 or X. Click here for more info: https://t.co/0zzgy1hdHv" #SprintCare


# Agent Evaluation 

In [15]:
import pandas as pd

golden_set = [
    {"message": whole_embeddings[0]['text'], "expected_intent": "General Enquiry", "should_escalate": False},
    {"message": whole_embeddings[1]['text'], "expected_intent": "General Enquiry", "should_escalate": False},
    {"message": whole_embeddings[2]['text'], "expected_intent": "Network Issue", "should_escalate": True},
    {"message": whole_embeddings[3]['text'], "expected_intent": "Network Issue", "should_escalate": True},
    {"message": whole_embeddings[4]['text'], "expected_intent": "General Enquiry", "should_escalate": False},
    {"message": whole_embeddings[5]['text'], "expected_intent": "Device Upgradation", "should_escalate": True},
    {"message": whole_embeddings[6]['text'], "expected_intent": "Cancellation", "should_escalate": True},
    {"message": whole_embeddings[7]['text'], "expected_intent": "General Enquiry", "should_escalate": False},
    {"message": whole_embeddings[8]['text'], "expected_intent": "Network Issue", "should_escalate": True},
    {"message": whole_embeddings[9]['text'], "expected_intent": "Network Issue", "should_escalate": True},
    {"message": whole_embeddings[10]['text'], "expected_intent": "Network Issue", "should_escalate": True},
]

def evaluate_agent():
    results = []
    for item in golden_set:
        output = run_agent(item["message"])
        intent_match = output["intent"].lower() == item["expected_intent"].lower()
        escalate_match = (output["action"] == "ESCALATE") == item["should_escalate"]
        results.append({
            "message": item["message"],
            "intent_match": intent_match,
            "escalate_match": escalate_match,
            "action": output["action"]
        })
    
    df_eval = pd.DataFrame(results)
    intent_accuracy = df_eval["intent_match"].mean() * 100
    escalation_accuracy = df_eval["escalate_match"].mean() * 100
    print(f"Evaluation Metrics -> Intent Accuracy: {intent_accuracy}% | Escalation Accuracy: {escalation_accuracy}%")

# Execute evaluation harness check
evaluate_agent()

Evaluation Metrics -> Intent Accuracy: 54.54545454545454% | Escalation Accuracy: 36.36363636363637%
